# 04 — Courbe risque–couverture : la preuve du « je ne sais pas »

**Rôle** : montrer que l'indice de fiabilité **prédit les erreurs**. Si on accepte de refuser les cas les moins fiables, la précision sur les cas conservés monte.

**Méthode** : on exécute le vrai pipeline (`MedGuardPipeline.analyze`) sur les splits `dev` et `test`.
- le seuil de refus est **réglé sur `dev`** ;
- les résultats sont **rapportés sur `test`** (jamais utilisé pour choisir quoi que ce soit).

**Entrées** : `calibration.json`, `operating_points.json`, `splits.json`. **Sorties** : `thresholds.json` (seuil de refus), `risk_coverage.png`.

Un cas est **correct** si toutes les pathologies évaluables sont bien classées au seuil de Youden. Les images rejetées avant le modèle (mauvais type d'examen, qualité insuffisante) sont comptées à part : ce sont des refus, pas des erreurs.

In [ ]:
# --- Initialisation (identique dans tous les notebooks) ---
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
IN_KAGGLE = Path("/kaggle").exists()
if IN_KAGGLE:
    # Internet doit être ACTIVÉ dans les réglages du notebook Kaggle.
    # Remplacez VOTRE_COMPTE par votre identifiant GitHub (dépôt medicure-ai).
    %pip install -q git+https://github.com/VOTRE_COMPTE/medicure-ai.git torchxrayvision
else:
    sys.path.insert(0, str(Path.cwd().parent))  # notebook lancé depuis le dossier notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from medguard.config import TARGETS, MODEL_WEIGHTS, Settings, load_settings, save_settings
from medguard.paths import find_file

# --- Configuration (à adapter) ---
DATASET = os.environ.get("MEDGUARD_DATASET", "nih")            # "nih" ou "chexpert"
DATA_ROOT = Path(os.environ.get("MEDGUARD_DATA", "/kaggle/input/data"))  # À ADAPTER au dataset attaché
WEIGHTS = os.environ.get("MEDGUARD_WEIGHTS", MODEL_WEIGHTS)
N_MAX = int(os.environ.get("MEDGUARD_NMAX", 6000))               # nb max d'images chargées
SEED = 42

WORK = Path("/kaggle/working") if IN_KAGGLE else Path(os.environ.get("MEDGUARD_WORK", "../outputs")).resolve()
ARTIFACTS = WORK / "artifacts"
FIGURES = WORK / "figures"
for d in (ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
print("Kaggle :", IN_KAGGLE, "| dataset :", DATASET, "| poids :", WEIGHTS, "| sortie :", WORK)

In [ ]:
import json
from medguard.calibration import Calibrator
from medguard.data import load_dataset
from medguard.evaluation import risk_coverage_curve, aurc, selective_metrics, threshold_for_coverage, bootstrap_ci
from medguard.model import get_predictor
from medguard.pipeline import MedGuardPipeline
from medguard.splits import load_splits

N_PER_SPLIT = int(os.environ.get("MEDGUARD_NEVAL", 1500))   # images analysées par split (6 passes modèle chacune)
TARGET_COVERAGE = 0.85                                       # part de cas sur lesquels on accepte de conclure

df, EVAL_TARGETS = load_dataset(DATASET, DATA_ROOT, n_max=N_MAX, seed=SEED)
splits = load_splits(find_file("splits.json", [WORK, ARTIFACTS]))
id_to_split = {i: name for name, ids in splits.items() for i in ids}
df["split"] = df.image_id.astype(str).map(id_to_split)
df = df.dropna(subset=["split"]).reset_index(drop=True)

try:
    settings = load_settings(find_file("thresholds.json", [ARTIFACTS, WORK]))
except FileNotFoundError:
    settings = load_settings()
calibrator = Calibrator.load(find_file("calibration.json", [ARTIFACTS, WORK]))
op_points = json.loads(find_file("operating_points.json", [ARTIFACTS, WORK]).read_text())
pipe = MedGuardPipeline(get_predictor(WEIGHTS), calibrator=calibrator, settings=settings)
print(pipe.predictor.info, "\ncalibration :", calibrator.fitted_on)

## 1. Analyser les images avec le vrai pipeline

In [ ]:
from tqdm.auto import tqdm

def run_split(name):
    d = df[df.split == name]
    d = d if len(d) <= N_PER_SPLIT else d.sample(N_PER_SPLIT, random_state=SEED)
    rows = []
    for r in tqdm(d.to_dict("records"), desc=name):
        res = pipe.analyze(r["path"], with_explanations=False)
        row = {"image_id": r["image_id"], "status": res.status,
               "reliability": res.reliability.index if res.reliability else np.nan,
               "quality": res.quality.score if res.quality else np.nan}
        if res.reliability:
            row.update({f"comp_{k}": v for k, v in res.reliability.components.items()})
        for t in res.targets:
            row[f"p_{t.name}"] = t.calibrated
        for t in EVAL_TARGETS:
            row[f"y_{t}"] = r[t]
        rows.append(row)
    return pd.DataFrame(rows)

res_dev, res_test = run_split("dev"), run_split("test")
for name, r in (("dev", res_dev), ("test", res_test)):
    print(name, r.status.value_counts().to_dict())

## 2. Un cas est-il correct ?

In [ ]:
def add_correct(r):
    r = r[r[f"p_{EVAL_TARGETS[0]}"].notna()].copy()      # le modèle a tourné
    ok = np.ones(len(r), dtype=bool)
    for t in EVAL_TARGETS:
        thr = op_points[t]["youden"]
        y = r[f"y_{t}"].values
        valid = ~np.isnan(y)
        pred = r[f"p_{t}"].values >= thr
        ok &= np.where(valid, pred == (y == 1), True)
    r["correct"] = ok
    return r

ev_dev, ev_test = add_correct(res_dev), add_correct(res_test)
print(f"cas évalués : dev {len(ev_dev)}, test {len(ev_test)} | exactitude globale test : {ev_test.correct.mean():.3f}")

## 3. Courbe risque–couverture

On trie les cas du plus au moins fiable. À couverture 100 % on garde tout ; en refusant de plus en plus de cas, le **risque** (taux d'erreur parmi les cas conservés) doit **baisser**. Comparaison avec un ordre aléatoire (aucune information) et avec chaque composante seule (ablation).

In [ ]:
rng = np.random.default_rng(SEED)
orders = {
    "indice de fiabilité (complet)": ev_test.reliability.values,
    "netteté de la décision seule": ev_test.comp_decisiveness.values,
    "qualité d'image seule": ev_test.comp_quality.values,
    "stabilité seule": ev_test.comp_stability.values + rng.normal(0, 1e-6, len(ev_test)),  # départage les ex aequo
    "ordre aléatoire": rng.random(len(ev_test)),
}
fig, ax = plt.subplots(figsize=(7, 4.5))
table = []
for name, rel in orders.items():
    curve = risk_coverage_curve(ev_test.correct.values, rel)
    ax.plot(curve["coverage"], curve["risk"], label=name, lw=2.4 if "complet" in name else 1.3, ls="--" if "aléatoire" in name else "-")
    table.append({"ordre": name, "AURC (plus bas = mieux)": aurc(ev_test.correct.values, rel)})
ax.set_xlabel("couverture (part de cas sur lesquels le système conclut)"); ax.set_ylabel("risque (taux d'erreur sur les cas conservés)")
ax.set_title("Courbe risque–couverture (split test)"); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIGURES / "risk_coverage.png", dpi=170); plt.show()
display(pd.DataFrame(table).round(4))

## 4. Régler le seuil de refus sur `dev`, le rapporter sur `test`

In [ ]:
t_star = float(np.clip(threshold_for_coverage(ev_dev.reliability.values, TARGET_COVERAGE), 30.0, 70.0))
print(f"Seuil de refus retenu (réglé sur dev, couverture visée {TARGET_COVERAGE:.0%}) : {t_star:.1f}")

m = selective_metrics(ev_test.correct.values, ev_test.reliability.values, t_star)
est, lo, hi = bootstrap_ci(lambda c, r: selective_metrics(c, r, t_star)["accuracy_kept"],
                           ev_test.correct.values, ev_test.reliability.values, n_boot=500)
report = pd.DataFrame([{
    "seuil": round(t_star, 1), "couverture (cas évalués)": round(m["coverage"], 3),
    "exactitude sans refus": round(m["accuracy_all"], 3),
    "exactitude avec refus": f"{est:.3f} [{lo:.3f}–{hi:.3f}]", "cas refusés": m["n_refused"]}])
display(report)

n_all = len(res_test)
print(f"Sur l'ensemble des {n_all} images du test : "
      f"{(res_test.status == 'unsupported').mean():.1%} hors périmètre, "
      f"{((res_test.status == 'refused') & res_test['p_' + EVAL_TARGETS[0]].isna()).mean():.1%} refusées pour qualité, "
      f"{((res_test.status == 'refused') & res_test['p_' + EVAL_TARGETS[0]].notna()).mean():.1%} refusées après analyse (confiance insuffisante).")

In [ ]:
APPLY_THRESHOLD = True
if APPLY_THRESHOLD:
    settings.reliability_refuse = round(t_star, 1)
    settings.reliability_high = max(settings.reliability_high, settings.reliability_refuse + 10)
    print("Écrit :", save_settings(settings, ARTIFACTS / "thresholds.json"))

### À dire au jury
> « Sans refus, le système est juste dans X % des cas. En acceptant de ne pas conclure sur Y % des cas (les moins fiables selon notre indice), il monte à Z % sur les cas restants. Un ordre aléatoire n'améliore rien : l'indice détecte réellement les cas à risque. »

Si la courbe complète n'est **pas** meilleure que l'ordre aléatoire, dites-le : c'est un résultat honnête, et il indique qu'il faut revoir la définition de l'indice (poids dans `config.py`), pas embellir les chiffres.